# 🧪 Validator MCP Server — Test Notebook

Test interactif des fonctions du validator MCP server.  
Les fonctions sont importées directement depuis `_internals.py` (pas via le protocole MCP).

**Ordre recommandé :**
1. Setup & connexion
2. Tester le cycle de vie du conteneur
3. Tester chaque tool individuellement
4. Lancer une suite complète
5. Voir le rapport final

## 1. Import & Configuration

In [1]:
import sys, json, time
from pathlib import Path
from IPython.display import display
import pandas as pd

# ── Ajouter le validator server au path ──────────────────────────────────────
VALIDATOR_DIR = Path(r"d:\OneDrive - Murex\Documents\interface_generation\mcp_servers\validator_server")
if str(VALIDATOR_DIR) not in sys.path:
    sys.path.insert(0, str(VALIDATOR_DIR))

from _internals import (
    WORKSPACE_HOST, DATA_HOST, CONTAINER_NAME, MAVEN_IMAGE,
    WORKSPACE_CONTAINER, DATA_CONTAINER,
    container_running, start_container, stop_container,
    docker_exec, resolve_jar_container,
    parse_compile_errors, parse_crash,
    load_test_config, resolve_suite, run_jar_and_diff,
    json_diff, json_score,
)

# ── Helpers d'affichage ───────────────────────────────────────────────────────
def ok(label):  print(f"✅ {label}")
def err(label): print(f"❌ {label}")
def section(title): print(f"\n{'─'*60}\n{title}\n{'─'*60}")

def show_score(result: dict):
    """Affiche le score et les détails d'un résultat run_jar_and_diff."""
    score  = result.get("score", 0.0)
    detail = result.get("score_detail", {})
    bar    = "█" * int(score // 5) + "░" * (20 - int(score // 5))
    grade  = "A" if score >= 90 else "B" if score >= 75 else "C" if score >= 60 else "D" if score >= 40 else "F"
    print(f"\nScore  : {score:5.1f}/100  [{bar}]  ({grade})")
    print(f"         {detail.get('matched', '?')}/{detail.get('total', '?')} feuilles correctes")
    if detail.get("missing"):
        print(f"Missing ({len(detail['missing'])})  : {', '.join(detail['missing'][:5])}")
    if detail.get("wrong_values"):
        for wv in detail["wrong_values"][:3]:
            print(f"  ≠ {wv['path']}: got '{wv['got']}' want '{wv['want']}'")
    if detail.get("extra"):
        print(f"Extra   ({len(detail['extra'])}) : {', '.join(detail['extra'][:5])}")

print("Imports OK")
print(f"   workspace host : {WORKSPACE_HOST}")
print(f"   data host      : {DATA_HOST}")
print(f"   container name : {CONTAINER_NAME}")


Imports OK
   workspace host : D:\OneDrive - Murex\Documents\interface_generation\workspaces
   data host      : D:\OneDrive - Murex\Documents\interface_generation\data\test
   container name : mcp-java-validator


## 2. Cycle de vie du conteneur Docker

In [2]:
section("Container — start")

result = start_container()
print(json.dumps(result, indent=2))

if result["ok"]:
    ok(f"Conteneur démarré  (note: {result.get('note') or result.get('container_id', '')})")
else:
    err(f"Échec démarrage : {result.get('stderr') or result.get('note')}")


────────────────────────────────────────────────────────────
Container — start
────────────────────────────────────────────────────────────
{
  "ok": true,
  "note": "already running"
}
✅ Conteneur démarré  (note: already running)


In [3]:
section("Container — status & smoke test")

running = container_running()
print(f"container_running() = {running}")

if running:
    ok("Conteneur vivant")
    # Vérifie que les volumes sont bien montés
    for path, label in [(WORKSPACE_CONTAINER, "workspace"), (DATA_CONTAINER, "data (read-only)")]:
        r = docker_exec(["ls", path])
        if r["ok"]:
            ok(f"Volume {label} accessible ({path})")
            print("   ", r["stdout"][:120].replace("\n", " | "))
        else:
            err(f"Volume {label} inaccessible : {r['stderr']}")
else:
    err("Conteneur non démarré — relancer la cellule précédente")


────────────────────────────────────────────────────────────
Container — status & smoke test
────────────────────────────────────────────────────────────
container_running() = True
✅ Conteneur vivant
✅ Volume workspace accessible (/workspace)
    cdm_news.txt | containers | Dockerfile | pom.xml | README.md | src | target | test_config.yaml | 
✅ Volume data (read-only) accessible (/data)
    bond-options-5-10-incomplete | bond-options-5-13 | commodity-5-10 | commodity-5-12 | commodity-derivatives-5-10-incomplete | commo


## 3. Test des tools individuels

In [4]:
section("Tool: compile_project")

t0 = time.time()
r  = docker_exec(["mvn", "clean", "compile", "-f", f"{WORKSPACE_CONTAINER}/pom.xml"])
elapsed = time.time() - t0
output  = r["stdout"] + r["stderr"]
errors  = parse_compile_errors(output)

print(f"Durée   : {elapsed:.1f}s")
print(f"Résultat: {'SUCCESS ✅' if r['ok'] else 'FAILURE ❌'}")
print(f"Erreurs : {len(errors)}")

if errors:
    for e in errors[:5]:
        if "file" in e:
            print(f"\n  {e['file']}:{e['line']} — {e['message']}")
            if e.get("snippet"):
                print(e["snippet"])
        else:
            print(f"\n  {e['message']}")
else:
    ok("Aucune erreur de compilation")


────────────────────────────────────────────────────────────
Tool: compile_project
────────────────────────────────────────────────────────────
Durée   : 16.1s
Résultat: SUCCESS ✅
Erreurs : 0
✅ Aucune erreur de compilation


In [5]:
section("Tool: list_test_suites")

config = load_test_config()
active = config.get("active_suite")
suites = config.get("suites", {})

print(f"active_suite : {active}")
print(f"Suites disponibles :\n")

rows = []
for name, suite in suites.items():
    pairs = resolve_suite(name)
    dirs  = suite.get("dirs", [])
    rows.append({
        "suite":       name,
        "active":      "✅" if name == active else "",
        "case_count":  len(pairs),
        "dirs":        dirs if dirs != "*" else ["* (all)"],
        "max_files":   suite.get("max_files_per_dir", "—"),
    })

df = pd.DataFrame(rows)
display(df)


────────────────────────────────────────────────────────────
Tool: list_test_suites
────────────────────────────────────────────────────────────
active_suite : equity_5_13_with_incomplete
Suites disponibles :



,suite,active,case_count,dirs,max_files
0,smoke,,8,"[equity-5-12, rates-5-10, credit-5-12, fx-5-12]",2
1,equity_5_13,,11,"[equity-5-12, equity-options-5-13, equity-swap...",—
2,equity_5_13_with_incomplete,✅,17,"[equity-5-12, equity-options-5-13, equity-opti...",—
3,v5_13_complete,,47,"[bond-options-5-13, commodity-derivatives-5-13...",—
4,all,,159,[* (all)],—


In [6]:
section("Tool: get_test_cases  (suite active)")

suite_name = config.get("active_suite") or list(suites.keys())[0]
pairs      = resolve_suite(suite_name)

print(f"Suite     : {suite_name}")
print(f"Cas total : {len(pairs)}\n")
print(f"{'FpML (container)':<60}  CDM (host, résumé)")
print("─" * 90)
for fpml, cdm in pairs[:10]:
    print(f"{fpml:<60}  ...{Path(cdm).name}")
if len(pairs) > 10:
    print(f"  ... ({len(pairs)-10} autres cas non affichés)")


────────────────────────────────────────────────────────────
Tool: get_test_cases  (suite active)
────────────────────────────────────────────────────────────
Suite     : equity_5_13_with_incomplete
Cas total : 17

FpML (container)                                              CDM (host, résumé)
──────────────────────────────────────────────────────────────────────────────────────────
/data/equity-5-12/fpml/eqs-ex01-single-underlyer-execution-long-form.xml  ...eqs-ex01-single-underlyer-execution-long-form.json
/data/equity-5-12/fpml/eqs-ex12-on-european-index-underlyer-short-form.xml  ...eqs-ex12-on-european-index-underlyer-short-form.json
/data/equity-5-12/fpml/eqs-ex15-forward-starting-pre-european-interdealer-share-swap-short-form.xml  ...eqs-ex15-forward-starting-pre-european-interdealer-share-swap-short-form.json
/data/equity-5-12/fpml/eqs-ex16-forward-starting-post-european-interdealer-share-swap-short-form.xml  ...eqs-ex16-forward-starting-post-european-interdealer-share-swap-sh

In [7]:
section("Tool: run_test  (1 cas)")

# Prend le premier cas disponible dans la suite active
if not pairs:
    err("Aucun cas dans la suite — vérifier test_config.yaml")
else:
    fpml_container, cdm_host = pairs[0]
    print(f"FpML : {fpml_container}")
    print(f"CDM  : {Path(cdm_host).name}\n")

    # Package + run
    t0  = time.time()
    pkg = docker_exec(["mvn", "package", "-DskipTests", "-q",
                       "-f", f"{WORKSPACE_CONTAINER}/pom.xml"], timeout=600)
    jar = resolve_jar_container()

    if not pkg["ok"]:
        err("mvn package a échoué")
        print(pkg["stderr"][-800:])
    elif not jar:
        err("JAR introuvable dans /workspace/target/")
    else:
        result  = run_jar_and_diff(jar, fpml_container, cdm_host)
        elapsed = time.time() - t0
        print(f"Durée  : {elapsed:.1f}s")
        print(f"Match  : {'✅ OUI' if result['match'] else '❌ NON'}")
        show_score(result)
        if result["crash"]:
            print(f"\nCrash  : {result['crash']['exception']} — {result['crash']['message']}")
            print(f"        at {result['crash']['method']} ({result['crash']['file']}:{result['crash']['line']})")
        if result["differences"] and not result["match"]:
            print(f"\nDifférences ({len(result['differences'])}) :")
            for d in result["differences"][:10]:
                print(f"  {d}")



────────────────────────────────────────────────────────────
Tool: run_test  (1 cas)
────────────────────────────────────────────────────────────
FpML : /data/equity-5-12/fpml/eqs-ex01-single-underlyer-execution-long-form.xml
CDM  : eqs-ex01-single-underlyer-execution-long-form.json

Durée  : 31.3s
Match  : ❌ NON

Score  :   0.0/100  [░░░░░░░░░░░░░░░░░░░░]  (F)
         0/245 feuilles correctes
Missing (11)  : trade.product, trade.tradeLot, trade.counterparty, trade.ancillaryParty, trade.adjustment

Différences (11) :
  MISSING trade.product  (expected: {"taxonomy": [{"source": "Other", "value": {"name": {"value": "Equity:Swap:Price)
  MISSING trade.tradeLot  (expected: [{"priceQuantity": [{"price": [{"value": {"value": 37.44, "unit": {"currency": {)
  MISSING trade.counterparty  (expected: [{"role": "Party1", "partyReference": {"globalReference": "33f59567", "externalR)
  MISSING trade.ancillaryParty  (expected: [{"role": "CalculationAgentIndependent", "partyReference": [{"globalRefe

In [8]:
section("Tool: run_arbitrary_test  (raw output, pas de diff)")

if pairs and jar:
    fpml_container, _ = pairs[0]
    result = docker_exec(["java", "-jar", jar, fpml_container], timeout=30)
    print(f"ok     : {result['ok']}")
    print(f"stdout : {result['stdout'][:500]}")
    if result["stderr"]:
        crash = parse_crash(result["stderr"])
        print(f"crash  : {crash}")
else:
    err("Pas de JAR ou de cas disponible")


────────────────────────────────────────────────────────────
Tool: run_arbitrary_test  (raw output, pas de diff)
────────────────────────────────────────────────────────────
ok     : True
stdout : {
  "trade" : { }
}



### Score détaillé — `json_score`

In [9]:
section("json_score — test unitaire")

# ── Test 1 : objet identique ────────────────────────────────────────────────
a_perfect = '{"trade": {"product": {"rate": 1.5}, "party": ["A", "B"]}}'
s = json_score(a_perfect, a_perfect)
print(f"[identique]  score={s['score']}  matched={s['matched']}/{s['total']}")
assert s["score"] == 100.0, f"Attendu 100, obtenu {s['score']}"
ok("score=100 sur JSON identique")

# ── Test 2 : objet vide (placeholder) ────────────────────────────────────────
a_placeholder = '{"trade": {}}'
e_rich        = '{"trade": {"product": {"rate": 1.5, "currency": "USD"}, "party": ["A", "B"]}}'
s = json_score(a_placeholder, e_rich)
print(f"[placeholder] score={s['score']}  matched={s['matched']}/{s['total']}")
print(f"  missing      : {s['missing']}")
assert s["score"] == 0.0, f"Attendu 0, obtenu {s['score']}"
ok("score=0 sur placeholder vide")

# ── Test 3 : partiel ─────────────────────────────────────────────────────────
a_partial = '{"trade": {"product": {"rate": 1.5}}}'
s = json_score(a_partial, e_rich)
print(f"[partiel]    score={s['score']}  matched={s['matched']}/{s['total']}")
print(f"  missing      : {s['missing']}")
print(f"  wrong_values : {s['wrong_values']}")
assert 0 < s["score"] < 100, f"Attendu entre 0 et 100, obtenu {s['score']}"
ok(f"score={s['score']} sur JSON partiel")

# ── Test 4 : sur le premier cas réel (si JAR dispo) ─────────────────────────
if 'jar' in dir() and jar and pairs:
    fpml_container, cdm_host = pairs[0]
    r     = run_jar_and_diff(jar, fpml_container, cdm_host)
    print(f"\n[premier cas réel]  FpML={Path(fpml_container).name}")
    show_score(r)
else:
    print("\n[cas réel] — lancer d'abord la cellule 'run_test 1 cas'")



────────────────────────────────────────────────────────────
json_score — test unitaire
────────────────────────────────────────────────────────────
[identique]  score=100.0  matched=3/3
✅ score=100 sur JSON identique
[placeholder] score=0.0  matched=0/4
  missing      : ['trade.product', 'trade.party']
✅ score=0 sur placeholder vide
[partiel]    score=25.0  matched=1/4
  missing      : ['trade.product.currency', 'trade.party']
  wrong_values : []
✅ score=25.0 sur JSON partiel

[premier cas réel]  FpML=eqs-ex01-single-underlyer-execution-long-form.xml

Score  :   0.0/100  [░░░░░░░░░░░░░░░░░░░░]  (F)
         0/245 feuilles correctes
Missing (11)  : trade.product, trade.tradeLot, trade.counterparty, trade.ancillaryParty, trade.adjustment


In [10]:
section("Tool: extract_method_source")

import re

def extract_method_source(java_source: str, method_name: str) -> dict:
    lines  = java_source.splitlines()
    sig_re = re.compile(
        r"(?:private|public|protected)\s+(?:static\s+)?(?:[\w<>\[\],\s]+)\s+"
        + re.escape(method_name) + r"\s*\("
    )
    start_idx = next((i for i, l in enumerate(lines) if sig_re.search(l)), None)
    if start_idx is None:
        return {"ok": False, "source": "", "note": f"Method '{method_name}' not found."}
    depth, in_body, out = 0, False, []
    for line in lines[start_idx:]:
        out.append(line)
        depth += line.count("{") - line.count("}")
        if depth > 0: in_body = True
        if in_body and depth <= 0: break
    return {"ok": True, "source": "\n".join(out), "note": ""}

# Trouver un fichier Java dans workspaces/src
java_files = list((WORKSPACE_HOST / "src").rglob("*.java"))
if not java_files:
    err("Aucun fichier .java trouvé dans workspaces/src")
else:
    java_file = java_files[0]
    source    = java_file.read_text(encoding="utf-8")
    print(f"Fichier : {java_file.name}")

    # Lister les méthodes disponibles
    methods = re.findall(
        r"(?:private|public|protected)\s+(?:static\s+)?[\w<>\[\],\s]+\s+(\w+)\s*\(", source
    )
    print(f"Méthodes trouvées : {methods[:10]}\n")

    if methods:
        result = extract_method_source(source, methods[0])
        if result["ok"]:
            ok(f"Méthode '{methods[0]}' extraite ({len(result['source'].splitlines())} lignes)")
            print(result["source"][:600])
        else:
            err(result["note"])


────────────────────────────────────────────────────────────
Tool: extract_method_source
────────────────────────────────────────────────────────────
Fichier : FileCopy.java
Méthodes trouvées : ['main']

✅ Méthode 'main' extraite (27 lignes)
    public static void main(String[] args) {
        if (args.length != 2) {
            System.err.println("Usage: FileCopy <input-file> <output-file>");
            System.exit(1);
        }

        Path input = Paths.get(args[0]);
        Path output = Paths.get(args[1]);

        if (!Files.exists(input)) {
            System.err.println("Erreur : le fichier source n'existe pas : " + input);
            System.exit(1);
        }

        try {
            byte[] content = Files.readAllBytes(input);
            // Crée les répertoires parents si nécessaire
            if (output.getParent(


## 4. Run Test Suite complète (`run_test_all`)

In [11]:
section(f"run_test_all — suite: {config.get('active_suite')}")

suite_name = config.get("active_suite")
pairs      = resolve_suite(suite_name)

if not pairs:
    err("Aucun cas — vérifier test_config.yaml")
else:
    jar = resolve_jar_container()
    if not jar:
        print("Packaging...")
        pkg = docker_exec(["mvn", "package", "-DskipTests", "-q",
                           "-f", f"{WORKSPACE_CONTAINER}/pom.xml"], timeout=600)
        if not pkg["ok"]:
            err("mvn package a échoué")
            print(pkg["stderr"][-600:])
        jar = resolve_jar_container()

    total, passed, failures, scores = 0, 0, [], []
    t0 = time.time()

    for fpml_container, cdm_host in pairs:
        total += 1
        r     = run_jar_and_diff(jar, fpml_container, cdm_host)
        score = r.get("score", 0.0)
        scores.append(score)
        bar   = "█" * int(score // 10) + "░" * (10 - int(score // 10))
        label = Path(fpml_container).name

        if r["match"]:
            passed += 1
            print(f"  ✅  {label:<45}  [{bar}] {score:5.1f}")
        else:
            crash_tag = f"  💥 {r['crash']['exception']}" if r["crash"] else f"  {len(r['differences'])} diffs"
            print(f"  ❌  {label:<45}  [{bar}] {score:5.1f}{crash_tag}")
            failures.append({
                "fpml":       label,
                "score":      score,
                "crash":      r["crash"]["exception"] if r["crash"] else None,
                "diff_count": len(r["differences"]),
                "first_diff": r["differences"][0][:80] if r["differences"] else "",
            })

    elapsed = time.time() - t0
    avg     = sum(scores) / len(scores) if scores else 0
    print(f"\nTerminé en {elapsed:.1f}s — {passed}/{total} passés — score moyen: {avg:.1f}/100")



────────────────────────────────────────────────────────────
run_test_all — suite: equity_5_13_with_incomplete
────────────────────────────────────────────────────────────
  ❌  eqs-ex01-single-underlyer-execution-long-form.xml  [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqs-ex12-on-european-index-underlyer-short-form.xml  [░░░░░░░░░░]   0.0  10 diffs
  ❌  eqs-ex15-forward-starting-pre-european-interdealer-share-swap-short-form.xml  [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqs-ex16-forward-starting-post-european-interdealer-share-swap-short-form.xml  [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex02-calendar-spread-short-form.xml        [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex13-1996-american-call-stock.xml          [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex04-european-call-index-long-form.xml     [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex11-quanto-long-form.xml                  [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex14-american-call-stock-passthrough-long-form.xml  [░░░░░░░░░░]   0.0  11 diffs
  ❌  eqd-ex20-ne

## 5. Rapport final

In [12]:
section("Rapport — résultats de la suite")

if 'total' not in dir() or total == 0:
    print("⚠️  Lancer la cellule 'run_test_all' d'abord.")
else:
    avg    = sum(scores) / len(scores) if scores else 0
    median = sorted(scores)[len(scores) // 2] if scores else 0

    # ── Résumé global ────────────────────────────────────────────────────────
    print(f"Suite      : {suite_name}")
    print(f"Total      : {total}")
    print(f"Passés     : {passed}  ({passed/total*100:.0f}%)")
    print(f"Échoués    : {total - passed}")
    print(f"Durée      : {elapsed:.1f}s")
    print(f"\nScore moyen   : {avg:.1f}/100")
    print(f"Score médian  : {median:.1f}/100")
    print(f"Score max     : {max(scores):.1f}/100")
    print(f"Score min     : {min(scores):.1f}/100")

    # ── Distribution des grades ───────────────────────────────────────────────
    def grade(s):
        return "A" if s >= 90 else "B" if s >= 75 else "C" if s >= 60 else "D" if s >= 40 else "F"

    from collections import Counter
    grade_counts = Counter(grade(s) for s in scores)
    print("\nDistribution des grades :")
    for g in ["A", "B", "C", "D", "F"]:
        count = grade_counts.get(g, 0)
        bar   = "█" * count
        print(f"  {g} : {bar:20s} {count:3d}")

    # ── Tableau des échecs ────────────────────────────────────────────────────
    if failures:
        print(f"\nDétail des {len(failures)} échecs :\n")
        df_fail = pd.DataFrame(failures).sort_values("score")
        df_fail.index = range(1, len(df_fail) + 1)
        display(df_fail.style.background_gradient(subset=["score"], cmap="RdYlGn", vmin=0, vmax=100))
    else:
        ok("Tous les tests sont passés 🎉")



────────────────────────────────────────────────────────────
Rapport — résultats de la suite
────────────────────────────────────────────────────────────
Suite      : equity_5_13_with_incomplete
Total      : 17
Passés     : 0  (0%)
Échoués    : 17
Durée      : 23.0s

Score moyen   : 0.0/100
Score médian  : 0.0/100
Score max     : 0.0/100
Score min     : 0.0/100

Distribution des grades :
  A :                        0
  B :                        0
  C :                        0
  D :                        0
  F : █████████████████     17

Détail des 17 échecs :



,fpml,score,crash,diff_count,first_diff
1,eqs-ex01-single-underlyer-execution-long-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""Other"", ""value"": {""n"
2,eqs-ex12-on-european-index-underlyer-short-form.xml,0.000000,None,10,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""ISDA"", ""productQuali"
3,eqs-ex15-forward-starting-pre-european-interdealer-share-swap-short-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""ISDA"", ""productQuali"
4,eqs-ex16-forward-starting-post-european-interdealer-share-swap-short-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""ISDA"", ""productQuali"
5,eqd-ex02-calendar-spread-short-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""Other"", ""value"": {""n"
6,eqd-ex13-1996-american-call-stock.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""Other"", ""value"": {""n"
7,eqd-ex04-european-call-index-long-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""Other"", ""value"": {""n"
8,eqd-ex11-quanto-long-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""ISDA"", ""productQuali"
9,eqd-ex14-american-call-stock-passthrough-long-form.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""Other"", ""value"": {""n"
10,eqd-ex20-nested-basket.xml,0.000000,None,11,"MISSING trade.product (expected: {""taxonomy"": [{""source"": ""ISDA"", ""productQuali"


## 6. LLM-as-Judge — `score_with_llm`

In [13]:
import os, httpx
from openai import OpenAI

VLLM_URL   = os.environ.get("VLLM_URL",   "http://10.27.40.184:8000/v1")
VLLM_MODEL = os.environ.get("VLLM_MODEL",  "/models/qwen3.6-27b-fp8")

def score_with_llm(actual_json: str, expected_json: str, fpml_context: str = "") -> dict:
    """Appelle le LLM-judge directement depuis le notebook."""
    context_block = f"\n\n### FpML input (reference)\n```xml\n{fpml_context[:3000]}\n```" if fpml_context else ""
    prompt = f"""You are an expert in the ISDA Common Domain Model (CDM).
Compare the following two CDM JSON objects and evaluate the quality of the mapping.

### Expected CDM JSON
```json
{expected_json[:4000]}
```

### Actual CDM JSON (produced by the converter)
```json
{actual_json[:4000]}
```{context_block}

Return a JSON object with exactly these fields:
{{
  "score": <integer 0-100>,
  "grade": "<A|B|C|D|F>",
  "summary": "<one paragraph>",
  "critical_issues": ["<issue 1>", ...],
  "minor_issues": ["<issue 1>", ...],
  "strengths": ["<strength 1>", ...]
}}

Grading scale: A=90-100, B=75-89, C=60-74, D=40-59, F=0-39.
Return ONLY the JSON object, no markdown fences, no preamble."""

    http_client = httpx.Client(timeout=httpx.Timeout(read=120.0, connect=10.0, write=30.0))
    client      = OpenAI(base_url=VLLM_URL, api_key="not-needed", http_client=http_client)
    response    = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    return json.loads(raw)

print(f"LLM endpoint : {VLLM_URL}")
print(f"Model        : {VLLM_MODEL}")


LLM endpoint : http://10.27.40.184:8000/v1
Model        : /models/qwen3.6-27b-fp8


In [14]:
section("LLM judge — sur le premier cas de la suite")

if 'jar' not in dir() or not jar or not pairs:
    print("⚠️  Lancer d'abord les cellules 'run_test 1 cas' ou 'run_test_all'")
else:
    fpml_container, cdm_host = pairs[0]
    r = run_jar_and_diff(jar, fpml_container, cdm_host)

    actual_json   = r["actual_json"]
    expected_json = Path(cdm_host).read_text(encoding="utf-8")

    # Score structurel (rapide, sans LLM)
    print("Score structurel (json_score) :")
    show_score(r)

    # Score LLM (plus lent, ~10-30s)
    print("\n⏳ Appel LLM-judge...")
    t0 = time.time()
    try:
        llm_result = score_with_llm(actual_json, expected_json)
        elapsed_llm = time.time() - t0

        print(f"\nLLM score : {llm_result['score']}/100  (grade {llm_result['grade']})  [{elapsed_llm:.1f}s]")
        print(f"\nSummary:\n  {llm_result['summary']}")
        if llm_result.get("critical_issues"):
            print(f"\nCritical issues ({len(llm_result['critical_issues'])}):")
            for iss in llm_result["critical_issues"]:
                print(f"  🔴 {iss}")
        if llm_result.get("minor_issues"):
            print(f"\nMinor issues ({len(llm_result['minor_issues'])}):")
            for iss in llm_result["minor_issues"]:
                print(f"  🟡 {iss}")
        if llm_result.get("strengths"):
            print(f"\nStrengths ({len(llm_result['strengths'])}):")
            for s in llm_result["strengths"]:
                print(f"  🟢 {s}")
    except Exception as e:
        err(f"LLM-judge a échoué : {e}")



────────────────────────────────────────────────────────────
LLM judge — sur le premier cas de la suite
────────────────────────────────────────────────────────────
Score structurel (json_score) :

Score  :   0.0/100  [░░░░░░░░░░░░░░░░░░░░]  (F)
         0/245 feuilles correctes
Missing (11)  : trade.product, trade.tradeLot, trade.counterparty, trade.ancillaryParty, trade.adjustment

⏳ Appel LLM-judge...
❌ LLM-judge a échoué : httpx.Timeout must either include a default, or set all four parameters explicitly.


In [15]:
# ── Nettoyer le conteneur (optionnel — décommenter si besoin) ─────────────────
# stop_container()
# print("Conteneur arrêté.")